# Sidewalk segmentation: train on Colab

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. Open https://huggingface.co/datasets/segments/sidewalk-semantic, log in and **accept the dataset conditions** (it is gated, CC BY-NC 4.0).
3. Run the cells top to bottom (~30-40 min for both models).

In [ ]:
!git clone https://github.com/gunstillrattles/sidewalk-segmentation-unet.git
%cd sidewalk-segmentation-unet
!pip -q install -r requirements.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste a read token from huggingface.co/settings/tokens

In [ ]:
# 1) U-Net from scratch (baseline)
!python src/train.py --model unet --epochs 60 --out checkpoints/unet --results results/unet
!python src/evaluate.py --ckpt checkpoints/unet/best.pt --results results/unet

In [ ]:
# 2) U-Net with a pretrained ResNet-34 encoder
!python src/train.py --model resnet34_unet --epochs 40 --out checkpoints/resnet34_unet --results results/resnet34_unet
!python src/evaluate.py --ckpt checkpoints/resnet34_unet/best.pt --results results/resnet34_unet

In [ ]:
import json
from IPython.display import Image, display
for name in ('unet', 'resnet34_unet'):
    m = json.load(open(f'results/{name}/metrics.json'))
    print(name, {k: m[k] for k in ('mIoU', 'pixel_accuracy', 'sidewalk_IoU')})
display(Image('results/resnet34_unet/sample_predictions.png'))

In [ ]:
# download results and weights, then commit results/ and attach checkpoints/*/best.pt to a GitHub Release
!zip -qr results.zip results
!cp checkpoints/resnet34_unet/best.pt resnet34_unet_best.pt && cp checkpoints/unet/best.pt unet_best.pt
from google.colab import files
files.download('results.zip'); files.download('resnet34_unet_best.pt'); files.download('unet_best.pt')